In [1]:
from datasets import load_dataset
from collections import Counter

In [2]:
dataset = load_dataset("Suryanshg/SUPER-NATURALINSTRUCTIONS-xlingual", streaming=True)


dataset

README.md: 0.00B [00:00, ?B/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['task_name', 'instruction', 'input_language', 'output_language', 'instruction_language', 'categories', 'input', 'output'],
        num_shards: 10
    })
    test: IterableDataset({
        features: ['task_name', 'instruction', 'input_language', 'output_language', 'instruction_language', 'categories', 'input', 'output'],
        num_shards: 1
    })
})

In [6]:
# Get Unique Training Tasks
train_tasks_only = dataset['train'].select_columns(['task_name'])
unique_train_tasks = set(item['task_name'] for item in train_tasks_only)

print("--- TRAIN SPLIT ---")
print(f"Total unique tasks: {len(unique_train_tasks)}")

# Get Unique Testing Tasks
test_tasks_only = dataset['test'].select_columns(['task_name'])
unique_test_tasks = set(item['task_name'] for item in test_tasks_only)

print("--- TEST SPLIT ---")
print(f"Total unique tasks: {len(unique_test_tasks)}")

--- TRAIN SPLIT ---
Total unique tasks: 1270
--- TEST SPLIT ---
Total unique tasks: 35


In [14]:
def find_tasks_with_multiple_outs_per_instance(split):
    seen_pairs = set()
    tasks_with_duplicates = set()

    # Only stream the columns we need to save bandwidth and parsing time
    train_subset = dataset[split].select_columns(['task_name', 'input'])

    # Iterate through the streaming dataset in batches for much better performance
    for batch in train_subset.iter(batch_size=1000):
        # Iterate over the zipped columns within the current batch
        for task, inp in zip(batch['task_name'], batch['input']):
            # Minor optimization: if we already know this task has duplicates, skip checking
            if task in tasks_with_duplicates:
                continue
                
            pair = (task, inp)
            if pair in seen_pairs:
                # If we've seen this exact task + input combination before, it's a duplicate
                tasks_with_duplicates.add(task)
            else:
                seen_pairs.add(pair)

    print(f"\n--- {split.upper()} SPLIT ---")
    print(f"Tasks with at least one duplicate input: {len(tasks_with_duplicates)}")
    return tasks_with_duplicates

train_tasks_with_duplicates = find_tasks_with_multiple_outs_per_instance("train")
test_tasks_with_duplicates = find_tasks_with_multiple_outs_per_instance("test")


--- TRAIN SPLIT ---
Tasks with at least one duplicate input: 112

--- TEST SPLIT ---
Tasks with at least one duplicate input: 3


### Regexes for Finding Instances with multiple outputs
```
"output":\s*\[\n\s*"[\w\s*]*",
"output":\s*\[\n\s*"[\w\s*?]*",
```


In [17]:
train_tasks_with_duplicates

{'task001_quoref_question_generation',
 'task002_quoref_answer_generation',
 'task023_cosmosqa_question_generation',
 'task024_cosmosqa_answer_generation',
 'task025_cosmosqa_incorrect_answer_generation',
 'task026_drop_question_generation',
 'task028_drop_answer_generation',
 'task043_essential_terms_answering_incomplete_questions',
 'task044_essential_terms_identifying_essential_words',
 'task045_miscellaneous_sentence_paraphrasing',
 'task046_miscellaneous_question_typing',
 'task059_ropes_story_generation',
 'task060_ropes_question_generation',
 'task067_abductivenli_answer_generation',
 'task068_abductivenli_incorrect_answer_generation',
 'task074_squad1.1_question_generation',
 'task075_squad1.1_answer_generation',
 'task082_babi_t1_single_supporting_fact_question_generation',
 'task1087_two_number_sum',
 'task108_contextualabusedetection_classification',
 'task111_asset_sentence_simplification',
 'task1217_atomic_answer_generation',
 'task127_scan_long_text_generation_action_com

In [18]:
test_tasks_with_duplicates

{'task396_persianqa_classification',
 'task463_parsinlu_entailment_classification',
 'task464_parsinlu_entailment_sentence_generation'}

### TODOs
1. Are all instructions in English?
2. What metric we need to use to figure out the quality of soft prompt elicitation? (Cosine Similarity with Contrastive Learning)
3. Viz distribution of Categories?
4. Easy vs Hard Tasks? Distribution?
5. Distribution of Hard prompt (Definition) length?